# NB09 — ML-Decoder baseline (strong multi-label head)

Reviewers will ask: *did you compare against a strong multi-label head, or only against
a linear classifier?* This notebook adds an **ML-Decoder**-style head (Ridnik et al.,
WACV 2023) on top of the same Swin-Tiny backbone, trained on the same five folds/seeds
with the same base loss (BCE + pos_weight), so the comparison is apples-to-apples
against the paper's Swin-Tiny baseline and Swin-Tiny+CCR.

The head is implemented from scratch (no fragile external dependency on Colab): a single
transformer-decoder layer with **K learnable group queries** (one per class) attending
over the backbone's spatial tokens, followed by a per-class projection to a logit. With
96 GB VRAM we can afford a wider decoder, but we keep it modest to stay a *fair* baseline
(the point is coverage, not winning by capacity).

**Interpretation either way is publishable:** if ML-Decoder wins, we report a stronger
ceiling; if it is unstable or ties at n≈1,990/5 labels, that *confirms* the paper's
narrative that complex heads need more data, and the low-cost CCR is justified.

## 0. Installation and imports

In [ ]:
import subprocess, sys
try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.12'])
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f'Ambiente: {"Google Colab" if IN_COLAB else "Local"}')

In [ ]:
import json, random, time
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import timm

from PIL import Image
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              average_precision_score, roc_auc_score)

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'timm    : {timm.__version__}')

## 1. Configuration

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Trilha3-V2')
    IMGS_DIR = Path('/content/Imgs')
    import os
    if not os.path.exists('/content/Imgs'):
        print('Descompactando imagens...')
        !unzip -q /content/drive/MyDrive/Trilha3-V2/Data/Imgs.zip -d /content/
        print('Descompactação concluída!')
else:
    BASE_DIR = Path('e:/anonymous-V3/Trilha3-V2')
    IMGS_DIR = Path('e:/anonymous-V3/Data/Imgs')

SPLITS_DIR  = BASE_DIR / 'splits'
RESULTS_DIR = BASE_DIR / 'results' / 'nb09'
CKPT_DIR    = BASE_DIR / 'checkpoints'
FIGS_DIR    = BASE_DIR / 'figs' / 'training'
NB03_RESULTS = BASE_DIR / 'results' / 'm2' / 'm2_results.json'  # baseline M0 + CCR p/ comparação
for d in [RESULTS_DIR, CKPT_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CORE_LABELS = ['ENANTEMA', 'PÓLIPO', 'ÚLCERA', 'EROSÃO', 'MICRONODULARIDADE']
N_FOLDS     = 5
N_CLASSES   = len(CORE_LABELS)

IMG_SIZE     = 224
BATCH_SIZE   = 128
MAX_EPOCHS   = 50
LR_BACKBONE  = 1e-4
LR_HEAD      = 1e-3
WEIGHT_DECAY = 1e-4
ES_PATIENCE  = 10
LR_PATIENCE  = 4
LR_FACTOR    = 0.5
SEEDS        = [42, 43, 44, 45, 46]
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Hiperparâmetros do ML-Decoder (modestos p/ baseline justo)
DEC_DIM      = 256   # dimensão do decoder (projeção das features do backbone)
DEC_HEADS    = 4     # cabeças de atenção
DEC_FFN      = 512   # feed-forward interno

print(f'Device   : {DEVICE}')
print(f'Decoder  : dim={DEC_DIM}, heads={DEC_HEADS}, ffn={DEC_FFN}, queries={N_CLASSES}')

## 2. Dataset, transforms, helpers (same as NB03/NB04)

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
def get_transforms(split):
    normalize = T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    if split == 'train':
        return T.Compose([
            T.Resize((int(IMG_SIZE*1.15), int(IMG_SIZE*1.15))),
            T.RandomHorizontalFlip(p=0.5), T.RandomRotation(degrees=10),
            T.RandomResizedCrop(size=IMG_SIZE, scale=(0.85,1.0), ratio=(0.9,1.1)),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
            T.ToTensor(), normalize])
    return T.Compose([
        T.Resize((int(IMG_SIZE*1.14), int(IMG_SIZE*1.14))),
        T.CenterCrop(IMG_SIZE), T.ToTensor(), normalize])

class GastroscopyDataset(Dataset):
    def __init__(self, df, imgs_dir, transform=None):
        self.df = df.reset_index(drop=True); self.imgs_dir = imgs_dir; self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.imgs_dir / row['image_name']).convert('RGB')
        labels = torch.tensor(row[CORE_LABELS].values.astype(float), dtype=torch.float32)
        if self.transform: img = self.transform(img)
        return img, labels

def compute_pos_weights(df):
    pos = df[CORE_LABELS].sum(axis=0).values
    neg = len(df) - pos
    return torch.tensor(neg/(pos+1e-6), dtype=torch.float32).to(DEVICE)

print('Dataset e helpers definidos.')

## 3. ML-Decoder head over Swin-Tiny

`SwinFeatures` exposes the backbone's spatial tokens via `forward_features` (shape
`(B, H, W, C)` for timm Swin), flattened to `(B, HW, C)`. The `MLDecoderHead` projects
tokens to `DEC_DIM`, runs **K class queries** through one cross-attention + FFN block,
and maps each refined query to its own logit via a grouped linear projection.

In [ ]:
class MLDecoderHead(nn.Module):
    """
    ML-Decoder simplificado (Ridnik et al., WACV 2023):
      - K group-queries aprendíveis (uma por classe)
      - 1 camada decoder: cross-attention (queries × tokens) + FFN
      - projeção por classe → logit
    """
    def __init__(self, in_dim, n_classes, dim=DEC_DIM, heads=DEC_HEADS, ffn=DEC_FFN):
        super().__init__()
        self.n_classes = n_classes
        self.input_proj = nn.Linear(in_dim, dim)
        self.queries    = nn.Parameter(torch.randn(n_classes, dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.ffn   = nn.Sequential(nn.Linear(dim, ffn), nn.GELU(), nn.Linear(ffn, dim))
        self.norm2 = nn.LayerNorm(dim)
        # projeção agrupada: cada query → 1 logit (pesos independentes por classe)
        self.group_fc = nn.Parameter(torch.randn(n_classes, dim) * 0.02)
        self.group_b  = nn.Parameter(torch.zeros(n_classes))

    def forward(self, tokens):              # tokens: (B, HW, in_dim)
        B = tokens.size(0)
        mem = self.input_proj(tokens)       # (B, HW, dim)
        q = self.queries.unsqueeze(0).expand(B, -1, -1)   # (B, K, dim)
        attn_out, _ = self.cross_attn(q, mem, mem)        # (B, K, dim)
        q = self.norm1(q + attn_out)
        q = self.norm2(q + self.ffn(q))                   # (B, K, dim)
        # logit por classe = <q_k, w_k> + b_k
        logits = (q * self.group_fc.unsqueeze(0)).sum(-1) + self.group_b   # (B, K)
        return logits

class SwinMLDecoder(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        self.backbone = timm.create_model(
            'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k',
            pretrained=True, num_classes=0)   # remove head; mantém forward_features
        feat_dim = self.backbone.num_features
        self.head = MLDecoderHead(feat_dim, n_classes)

    def forward(self, x):
        feats = self.backbone.forward_features(x)   # timm Swin: (B, H, W, C)
        if feats.dim() == 4:
            B, H, W, C = feats.shape
            tokens = feats.reshape(B, H * W, C)
        elif feats.dim() == 3:
            tokens = feats                          # já (B, HW, C)
        else:
            raise ValueError(f'forward_features shape inesperado: {feats.shape}')
        return self.head(tokens)

def build_optimizer(model):
    head = list(model.head.parameters())
    back = [p for n, p in model.named_parameters() if not n.startswith('head')]
    return torch.optim.AdamW([{'params': back, 'lr': LR_BACKBONE},
                              {'params': head, 'lr': LR_HEAD}], weight_decay=WEIGHT_DECAY)

# Sanidade de forma (1 batch fake)
_m = SwinMLDecoder().to(DEVICE)
_x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad():
    _out = _m(_x)
print(f'Saída do SwinMLDecoder: {tuple(_out.shape)}  (esperado (2, {N_CLASSES}))')
del _m, _x, _out
if torch.cuda.is_available(): torch.cuda.empty_cache()

## 4. Metrics (same as NB03/NB04)

In [ ]:
def optimize_thresholds(probs, labels):
    thr = np.zeros(N_CLASSES)
    for i in range(N_CLASSES):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.1, 0.9, 0.05):
            preds = (probs[:, i] >= t).astype(int)
            if preds.sum() == 0: continue
            f1 = f1_score(labels[:, i], preds, zero_division=0)
            if f1 > best_f1: best_f1, best_t = f1, t
        thr[i] = best_t
    return thr

def compute_metrics(probs, labels, thresholds=None):
    if thresholds is None: thresholds = np.full(N_CLASSES, 0.5)
    preds = (probs >= thresholds).astype(int)
    f1_per = f1_score(labels, preds, average=None, zero_division=0)
    pr_per = precision_score(labels, preds, average=None, zero_division=0)
    rc_per = recall_score(labels, preds, average=None, zero_division=0)
    auprc, rocauc = [], []
    for i in range(N_CLASSES):
        auprc.append(average_precision_score(labels[:,i], probs[:,i]) if labels[:,i].sum()>0 else float('nan'))
        rocauc.append(roc_auc_score(labels[:,i], probs[:,i]) if 0<labels[:,i].sum()<len(labels) else float('nan'))
    multi_mask = labels.sum(axis=1) >= 2
    f1_multi = float(f1_score(labels[multi_mask], preds[multi_mask], average='macro', zero_division=0)) if multi_mask.sum()>0 else float('nan')
    result = {'f1_macro': float(np.nanmean(f1_per)),
              'f1_micro': float(f1_score(labels, preds, average='micro', zero_division=0)),
              'hamming':  float(np.mean(preds != labels)), 'f1_multi': f1_multi,
              'pr_auc_macro': float(np.nanmean(auprc)), 'roc_auc_macro': float(np.nanmean(rocauc))}
    for i, label in enumerate(CORE_LABELS):
        result[f'f1_{label}']    = float(f1_per[i])
        result[f'prec_{label}']  = float(pr_per[i])
        result[f'rec_{label}']   = float(rc_per[i])
        result[f'auprc_{label}'] = float(auprc[i])
        result[f'thr_{label}']   = float(thresholds[i])
    return result

print('Métricas definidas.')

## 5. Training loop

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); probs, labels = [], []
    for imgs, lab in loader:
        probs.append(torch.sigmoid(model(imgs.to(DEVICE))).cpu().numpy())
        labels.append(lab.numpy())
    return np.concatenate(probs), np.concatenate(labels)

def train_one_fold(fold, save_checkpoint=True, verbose=True):
    set_seed(SEEDS[fold])
    df_tr  = pd.read_csv(SPLITS_DIR / f'fold_{fold}_train.csv')
    df_val = pd.read_csv(SPLITS_DIR / f'fold_{fold}_val.csv')
    df_te  = pd.read_csv(SPLITS_DIR / f'fold_{fold}_test.csv')
    ds_tr  = GastroscopyDataset(df_tr,  IMGS_DIR, get_transforms('train'))
    ds_val = GastroscopyDataset(df_val, IMGS_DIR, get_transforms('val'))
    ds_te  = GastroscopyDataset(df_te,  IMGS_DIR, get_transforms('val'))
    dl_tr  = DataLoader(ds_tr,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    dl_te  = DataLoader(ds_te,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    model     = SwinMLDecoder().to(DEVICE)
    pos_w     = compute_pos_weights(df_tr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)   # mesma base loss do baseline
    optimizer = build_optimizer(model)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max',
                                                          patience=LR_PATIENCE, factor=LR_FACTOR)
    best_val_f1, best_weights, es = -1.0, None, 0
    history = {'train_loss': [], 'val_f1': []}

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train(); running = 0.0
        for imgs, lab in dl_tr:
            imgs, lab = imgs.to(DEVICE), lab.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), lab)
            loss.backward(); optimizer.step()
            running += loss.item() * len(imgs)
        train_loss = running / len(ds_tr)
        vp, vl = evaluate(model, dl_val)
        vthr = optimize_thresholds(vp, vl)
        vf1  = compute_metrics(vp, vl, vthr)['f1_macro']
        scheduler.step(vf1)
        history['train_loss'].append(train_loss); history['val_f1'].append(vf1)
        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f'  Epoch {epoch:>3} | loss={train_loss:.4f} | val_F1={vf1:.4f}')
        if vf1 > best_val_f1 + 1e-4:
            best_val_f1, best_weights, es = vf1, deepcopy(model.state_dict()), 0
        else:
            es += 1
            if es >= ES_PATIENCE:
                if verbose: print(f'  Early stopping na época {epoch}.')
                break

    model.load_state_dict(best_weights)
    if save_checkpoint:
        torch.save(best_weights, CKPT_DIR / f'swin_mldecoder_fold{fold}.pt')
    tp, tl = evaluate(model, dl_te)
    met = compute_metrics(tp, tl, vthr)
    met.update({'fold': fold, 'best_val_f1': best_val_f1,
                'epochs_trained': len(history['train_loss']),
                'val_thresholds': vthr.tolist(), 'history': history,
                'test_preds': (tp >= vthr).astype(int).tolist(),
                'test_labels': tl.tolist()})
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return met

print('Loop de treino definido.')

## 6. Execution (with resume)

In [ ]:
RUN_FAST_CHECK = False
if RUN_FAST_CHECK:
    MAX_EPOCHS, FOLDS_TO_RUN = 5, [0]
    print('MODO FAST CHECK: 1 fold, 5 épocas.')
else:
    FOLDS_TO_RUN = list(range(N_FOLDS))
    print(f'MODO COMPLETO: {N_FOLDS} folds.')

results_file = RESULTS_DIR / 'mldecoder_results.json'
if results_file.exists():
    print('>> ENCONTRADO RESULTADO ANTERIOR! Retomando... <<')
    with open(results_file, encoding='utf-8') as f:
        all_results = json.load(f)
else:
    all_results = {}

exp_id = 'mldecoder'
done = all_results.get(exp_id, {}).get('fold_results', [])
done_folds = {r['fold'] for r in done}
fold_results = list(done)

t0 = time.time()
for fold in FOLDS_TO_RUN:
    if fold in done_folds:
        print(f'\n[PULANDO] Fold {fold} já concluído.')
        continue
    print(f'\n{"="*60}\nML-Decoder — Fold {fold}/{N_FOLDS-1}\n{"="*60}')
    met = train_one_fold(fold, save_checkpoint=not RUN_FAST_CHECK)
    fold_results.append(met)
    print(f'  → Test Macro-F1={met["f1_macro"]:.4f}  PÓLIPO={met["f1_PÓLIPO"]:.4f}  '
          f'MICRONOD={met["f1_MICRONODULARIDADE"]:.4f}')
    # salva incremental
    metric_keys = (['f1_macro','f1_micro','hamming','f1_multi','pr_auc_macro','roc_auc_macro'] +
                   [f'f1_{l}' for l in CORE_LABELS] + [f'auprc_{l}' for l in CORE_LABELS])
    agg = {}
    for k in metric_keys:
        vals = [r[k] for r in fold_results if not np.isnan(r.get(k, float('nan')))]
        if vals: agg[k] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals))}
    all_results[exp_id] = {'config': {'head': 'ml_decoder', 'dim': DEC_DIM,
                                      'heads': DEC_HEADS, 'ffn': DEC_FFN},
                          'fold_results': fold_results, 'aggregated': agg,
                          'elapsed_sec': round(time.time()-t0, 1)}
    with open(results_file, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print('  Salvo parcialmente.')

agg = all_results[exp_id]['aggregated']
print(f'\nAGREGADO ML-Decoder: Macro-F1 = {agg["f1_macro"]["mean"]:.4f} ± {agg["f1_macro"]["std"]:.4f}')

## 7. Comparison vs. Swin-Tiny baseline and Swin+CCR

Paired bootstrap (B=2000) on pooled 5-fold test predictions, against M0 (linear head)
and the real CCR (`m2_l06`) from NB03.

In [ ]:
with open(NB03_RESULTS, encoding='utf-8') as f:
    nb03 = json.load(f)

def collect_nb03(key):
    p, l = [], []
    for r in nb03[key]['fold_results']:
        p.extend(r['test_preds']); l.extend(r['test_labels'])
    return np.array(p), np.array(l)

def collect_local(exp_id):
    p, l = [], []
    for r in all_results[exp_id]['fold_results']:
        p.extend(r['test_preds']); l.extend(r['test_labels'])
    return np.array(p), np.array(l)

def bootstrap_compare(pa, la, pb, lb, B=2000, seed=42):
    rng = np.random.default_rng(seed); n = len(la); deltas = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        deltas.append(f1_score(lb[idx], pb[idx], average='macro', zero_division=0) -
                      f1_score(la[idx], pa[idx], average='macro', zero_division=0))
    deltas = np.array(deltas)
    return {'delta_mean': float(deltas.mean()),
            'ci_95_low': float(np.percentile(deltas, 2.5)),
            'ci_95_high': float(np.percentile(deltas, 97.5)),
            'p_value': float(np.mean(deltas <= 0)),
            'significant': bool(np.percentile(deltas, 2.5) > 0)}

pmd, lmd = collect_local('mldecoder')
f1_md = f1_score(lmd, pmd, average='macro', zero_division=0)
print(f'ML-Decoder (agrupado): F1={f1_md:.4f}\n')

comparisons = {}
print(f'{"vs":<22} {"F1 base":>8} {"Δ ML-Dec":>10} {"CI95":>20} {"p":>7} {"Sig":>5}')
print('-'*76)
for key, name in [('m0_swin', 'Swin linear (M0)'), ('m2_l06', 'Swin + CCR (λ=0.6)')]:
    if key not in nb03: continue
    pb, lb = collect_nb03(key)
    f1_b = f1_score(lb, pb, average='macro', zero_division=0)
    # Δ = ML-Decoder − base
    boot = bootstrap_compare(pb, lb, pmd, lmd)
    comparisons[f'mldecoder_vs_{key}'] = {**boot, 'f1_base': float(f1_b), 'f1_mldecoder': float(f1_md)}
    ci = f'[{boot["ci_95_low"]:+.3f}, {boot["ci_95_high"]:+.3f}]'
    sig = '✓' if boot['significant'] else '—'
    print(f'{name:<22} {f1_b:>8.4f} {boot["delta_mean"]:>+10.4f} {ci:>20} {boot["p_value"]:>7.3f} {sig:>5}')

with open(RESULTS_DIR / 'mldecoder_bootstrap.json', 'w', encoding='utf-8') as f:
    json.dump(comparisons, f, ensure_ascii=False, indent=2)
print('\nComparações salvas.')

## 8. Figure + table (EN + PT)

In [ ]:
m0  = nb03['m0_swin']['aggregated']
ccr = nb03['m2_l06']['aggregated']
mld = all_results['mldecoder']['aggregated']

MODELS = [('Swin linear\n(M0)', m0, '#90A4AE'),
          ('Swin + CCR\n(λ=0.6)', ccr, '#2E7D32'),
          ('Swin +\nML-Decoder', mld, '#6A1B9A')]
TITLE = {'en': 'Strong multi-label head comparison (Swin-Tiny)',
         'pt': 'Comparação com cabeça multi-label forte (Swin-Tiny)'}
YLAB  = {'en': 'Macro-F1 (mean ± sd, 5 folds)', 'pt': 'Macro-F1 (média ± dp, 5 folds)'}

def make_fig(lang):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    names = [m[0] for m in MODELS]
    means = [m[1]['f1_macro']['mean'] for m in MODELS]
    stds  = [m[1]['f1_macro']['std']  for m in MODELS]
    cols  = [m[2] for m in MODELS]
    bars = axes[0].bar(names, means, yerr=stds, color=cols, capsize=5,
                       edgecolor='white', linewidth=0.6)
    for b, m in zip(bars, means):
        axes[0].text(b.get_x()+b.get_width()/2, m+0.005, f'{m:.3f}', ha='center', va='bottom', fontsize=9)
    axes[0].set_ylabel(YLAB[lang]); axes[0].set_ylim(min(means)-0.05, max(means)+0.05)
    axes[0].set_title(TITLE[lang], fontweight='bold'); axes[0].grid(axis='y', alpha=0.3)
    heat = {m[0].replace(chr(10), ' '): [m[1].get(f'f1_{l}', {}).get('mean', float('nan')) for l in CORE_LABELS] for m in MODELS}
    df_heat = pd.DataFrame(heat, index=CORE_LABELS).T
    sns.heatmap(df_heat, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0.3, vmax=0.85,
                linewidths=0.4, ax=axes[1], cbar_kws={'label': 'F1'})
    axes[1].set_title('F1 / class' if lang=='en' else 'F1 / classe', fontweight='bold')
    plt.tight_layout()
    out = FIGS_DIR / f'nb09_mldecoder_{lang}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Figura salva: {out.name}')

for lang in ['en', 'pt']:
    make_fig(lang)

rows = []
for name, agg, _ in MODELS:
    row = {'Model': name.replace(chr(10), ' ')}
    for k in ['f1_macro','f1_ENANTEMA','f1_PÓLIPO','f1_ÚLCERA','f1_EROSÃO','f1_MICRONODULARIDADE']:
        v = agg.get(k)
        row[k.replace('f1_', '')] = f"{v['mean']:.3f}±{v['std']:.3f}" if v else '-'
    rows.append(row)
df_table = pd.DataFrame(rows).set_index('Model')
print('\n=== TABELA ===')
print(df_table.to_string())
df_table.to_csv(RESULTS_DIR / 'tabela_nb09.csv')